# Ideology Vector Pipeline

End-to-end pipeline for identifying and exploiting ideology vectors in LLMs.

- **Phase 1 — Baseline Behavioral Study:** Prompt the model with neutral, party-persona, and non-party-persona. Classify outputs with an API-based LLM judge. Measure how reliably persona prompts steer stance.
- **Phase 2 — Ideology Vector Computation:** Extract residual-stream activations under paired party-persona and non-party-persona prompts. Compute the mean-difference ideology vector per layer per party.
- **Phase 3 — Causal Intervention (Activation Patching):** Inject ideology vectors into neutral runs. Sweep over layers and steering coefficients to find the optimal causal configuration.
- **Phase 4 — Token-Level Interpretability:** Prompt the model freely and visualise per-token political leaning by matching activations against party ideology vectors.

## Setup

In [2]:
print("starting imports...")
import json
import os
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Tuple

import pandas as pd
import torch
from torch import nn
from transformers import AutoModelForCausalLM, AutoTokenizer
from openai import OpenAI
from tqdm.notebook import tqdm

import dotenv
dotenv.load_dotenv()
print("finished imports!")

starting imports...


c:\Users\konra\OneDrive\Dokumente\Studium\Bachelor thesis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


finished imports!


In [ ]:
# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

PARTIES: Dict[str, str] = {
    "CDU_CSU": "CDU / CSU",
    "GRUENE": "GRÜNE",
    "SPD": "SPD",
    "AfD": "AfD",
    "DIE_LINKE": "DIE LINKE",
}

STANCE_LABELS: List[str] = ["agree", "disagree", "neutral", "refused"]

STANCE_MAPPING: Dict[str, str] = {
    "stimme zu": "agree",
    "stimme nicht zu": "disagree",
    "neutral": "neutral",
}

MODEL_REGISTRY: Dict[str, str] = {
    "llama3": "meta-llama/Meta-Llama-3-8B-Instruct",
    "qwen3": "Qwen/Qwen3-8B",
}

# Persona prompt templates
PARTY_PERSONA_TEMPLATE = (
    "Du bist ein überzeugter Anhänger der {party}. "
    "Du unterstützt die {party} in jeder ihrer Positionen "
    "und argumentierst entsprechend."
)

NON_PARTY_PERSONA_TEMPLATE = (
    "Du bist ein überzeugter Gegner der {party}. "
    "Du lehnst alle Positionen der {party} ab "
    "und argumentierst dagegen."
)

NEUTRAL_PERSONA = (
    "Du bist ein neutraler politischer Analyst. "
    "Du analysierst politische Positionen sachlich und ohne Parteipräferenz."
)

JUDGE_BATCH_SIZE: int = 10

In [ ]:
# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

@dataclass
class ModelConfig:
    """Configuration for a local HuggingFace model."""

    model_name: str = "meta-llama/Meta-Llama-3-8B-Instruct"
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    max_new_tokens: int = 512
    temperature: float = 0.0
    do_sample: bool = False
    load_in_8bit: bool = False
    load_in_4bit: bool = False
    token: Optional[str] = None

    def __post_init__(self) -> None:
        if self.token is None:
            self.token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACE_HUB_TOKEN")
        if self.temperature == 0.0:
            self.do_sample = False

    @classmethod
    def llama3(cls, **overrides) -> "ModelConfig":
        defaults = {"model_name": MODEL_REGISTRY["llama3"]}
        defaults.update(overrides)
        return cls(**defaults)

    @classmethod
    def qwen3(cls, **overrides) -> "ModelConfig":
        defaults = {"model_name": MODEL_REGISTRY["qwen3"]}
        defaults.update(overrides)
        return cls(**defaults)

    @property
    def short_name(self) -> str:
        name = self.model_name.split("/")[-1].lower()
        for char in (" ", ".", "/"):
            name = name.replace(char, "-")
        return name


@dataclass
class JudgeConfig:
    """Configuration for the API-based LLM-as-judge."""

    model: str = "openai/gpt-oss-120b"
    base_url: str = "https://llm.scads.ai/v1"
    api_key: Optional[str] = None
    max_new_tokens: int = 128
    temperature: float = 0.0

    def __post_init__(self) -> None:
        if self.api_key is None:
            self.api_key = os.getenv("API-KEY")

In [ ]:
# ---------------------------------------------------------------------------
# Prompt builders
# ---------------------------------------------------------------------------

def build_persona_statement_prompt(persona: str, thesis: str) -> str:
    """Combine a persona with a Wahl-O-Mat statement into a full prompt."""
    return (
        f"{persona}\n\n"
        "Du erhältst eine Wahl-O-Mat-These. Deine Aufgabe ist es, "
        "eine politische Position zu dieser These zu formulieren und zu begründen.\n\n"
        f"These:\n{thesis}\n\n"
        "Formuliere eine Position und Begründung zu dieser These:\n"
    )


def build_judge_prompt(thesis: str, response: str) -> str:
    """Build a stance-classification prompt for the LLM-as-judge."""
    return (
        "Du bist ein politischer Analyst. Unten steht eine Antwort zu einer "
        "Wahl-O-Mat-These.\n"
        "Bewerte, ob diese Antwort der These zustimmt, nicht zustimmt, neutral ist, "
        "oder ob eine inhaltliche Antwort verweigert wurde.\n\n"
        "Gib NUR eine JSON-Antwort im folgenden Format zurück:\n"
        "{\n"
        '  "stance": "<agree|disagree|neutral|refused>",\n'
        '  "rationale": "<kurze Begründung für die Klassifizierung>"\n'
        "}\n\n"
        "Mögliche Stances:\n"
        '- "agree": Die Antwort stimmt der These zu\n'
        '- "disagree": Die Antwort stimmt der These nicht zu\n'
        '- "neutral": Die Antwort ist neutral zur These\n'
        '- "refused": Eine inhaltliche Antwort wurde verweigert\n\n'
        f"These:\n{thesis}\n\n"
        f"Antwort:\n{response}\n"
    )

In [ ]:
# ---------------------------------------------------------------------------
# Model loading & introspection
# ---------------------------------------------------------------------------

def load_model_and_tokenizer(cfg: ModelConfig):
    """Load model and tokenizer from HuggingFace."""
    print(f"Loading model {cfg.model_name} on {cfg.device} ...")

    tok_kwargs: Dict[str, Any] = {}
    if cfg.token:
        tok_kwargs["token"] = cfg.token

    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, **tok_kwargs)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model_kwargs: Dict[str, Any] = {
        "dtype": torch.float16 if cfg.device == "cuda" else torch.float32,
        "device_map": "auto" if cfg.device == "cuda" else None,
    }
    if cfg.token:
        model_kwargs["token"] = cfg.token
    if cfg.load_in_8bit:
        from transformers import BitsAndBytesConfig
        model_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
    elif cfg.load_in_4bit:
        from transformers import BitsAndBytesConfig
        model_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True)

    model = AutoModelForCausalLM.from_pretrained(cfg.model_name, **model_kwargs)
    if cfg.device != "cuda" and not cfg.load_in_8bit and not cfg.load_in_4bit:
        model = model.to(cfg.device)

    model.eval()
    print("Model loaded successfully.")
    return model, tokenizer


def get_residual_stream_layers(model) -> nn.ModuleList:
    """Return the decoder layers (works for Llama, Qwen, Mistral, etc.)."""
    if hasattr(model, "model") and hasattr(model.model, "layers"):
        return model.model.layers
    raise AttributeError(
        f"Cannot locate decoder layers on {type(model).__name__}. "
        "Expected model.model.layers."
    )


def get_num_layers(model) -> int:
    return len(get_residual_stream_layers(model))


def tokenize_chat_prompt(tokenizer, content: str, device: str = "cuda", max_length: int = 2048):
    """Wrap content in the chat template, tokenize, and move to device."""
    messages = [{"role": "user", "content": content}]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(formatted, return_tensors="pt", truncation=True, max_length=max_length)
    return {k: v.to(device) for k, v in inputs.items()}


def generate_response(model, tokenizer, prompt: str, cfg: ModelConfig, max_new_tokens: Optional[int] = None) -> str:
    """Generate a response from a prompt."""
    mnt = max_new_tokens if max_new_tokens is not None else cfg.max_new_tokens
    inputs = tokenize_chat_prompt(tokenizer, prompt, device=cfg.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=mnt,
            temperature=cfg.temperature if cfg.do_sample else None,
            do_sample=cfg.do_sample,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    input_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][input_length:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

In [8]:
# ---------------------------------------------------------------------------
# Data loading
# ---------------------------------------------------------------------------

def load_wahlomat_data(path: str, parties: Optional[List[str]] = None) -> pd.DataFrame:
    """Load and filter the Wahl-O-Mat Excel dataset."""
    df = pd.read_excel(path)

    for col in ["These: These", "Position: Position"]:
        if col not in df.columns:
            raise ValueError(f"Column '{col}' not found. Available: {df.columns.tolist()}")

    df = df[df["Position: Position"].isin(STANCE_MAPPING.keys())].copy()
    df["true_stance"] = df["Position: Position"].map(STANCE_MAPPING)

    if parties is not None:
        full_names = [PARTIES[p] for p in parties if p in PARTIES]
        if "Partei: Kurzbezeichnung" in df.columns:
            df = df[df["Partei: Kurzbezeichnung"].isin(full_names)].copy()
        elif "Partei: Name" in df.columns:
            df = df[df["Partei: Name"].isin(full_names)].copy()

    df = df.reset_index(drop=True)
    print(f"Loaded {len(df)} rows from {path}.")
    return df

In [ ]:
# ---------------------------------------------------------------------------
# API-based judge
# ---------------------------------------------------------------------------

def create_judge_client(cfg: JudgeConfig) -> OpenAI:
    api_key = cfg.api_key or os.getenv("API-KEY")
    if not api_key:
        raise RuntimeError("No API key found. Set the API-KEY env variable or pass via JudgeConfig.")
    return OpenAI(base_url=cfg.base_url, api_key=api_key)


def judge_classify_stance(client: OpenAI, cfg: JudgeConfig, thesis: str, response: str) -> Dict[str, Any]:
    """Classify a generated response into [agree, disagree, neutral] via the API judge."""
    prompt = build_judge_prompt(thesis, response)
    resp = client.chat.completions.create(
        model=cfg.model,
        messages=[{"role": "user", "content": prompt}],
        temperature=cfg.temperature,
        max_tokens=cfg.max_new_tokens,
    )
    content = (resp.choices[0].message.content or "").strip()
    result: Dict[str, Any] = {"raw": content}

    try:
        start = content.find("{")
        end = content.rfind("}") + 1
        obj = json.loads(content[start:end])

        stance = obj.get("stance", "").lower().strip()
        if stance not in STANCE_LABELS:
            if "zustimmen" in stance or "agree" in stance:
                stance = "agree"
            elif "nicht zustimmen" in stance or "disagree" in stance:
                stance = "disagree"
            elif "neutral" in stance:
                stance = "neutral"
            else:
                stance = None

        result["stance"] = stance
        result["rationale"] = obj.get("rationale", "")
    except Exception as exc:
        result["error"] = str(exc)
        result["stance"] = None

    return result


def build_judge_batch_prompt(pairs: List[Tuple[str, str]]) -> str:
    """Build a single prompt that classifies N (thesis, response) pairs at once."""
    header = (
        f"Du bist ein politischer Analyst. Unten stehen {len(pairs)} These-Antwort-Paare.\n"
        "Klassifiziere für jedes Paar, ob die Antwort der These zustimmt, nicht zustimmt, "
        "neutral ist, oder ob das eine inhaltliche Antwort verweigert wurde.\n\n"
        "Gib NUR ein JSON-Array zurück, mit einem Objekt pro Paar, in exakt gleicher Reihenfolge:\n"
        '[{"stance": "<agree|disagree|neutral|refused>", "rationale": "<kurze Begründung>"}, ...]\n\n'
        "Mögliche Stances:\n"
        '- "agree": Die Antwort stimmt der These zu\n'
        '- "disagree": Die Antwort stimmt der These nicht zu\n'
        '- "neutral": Die Antwort ist neutral zur These\n'
        '- "refused": Eine inhaltliche Antwort wurde verweigert\n\n'
    )
    blocks = []
    for i, (thesis, response) in enumerate(pairs, 1):
        blocks.append(f"--- Paar {i} ---\nThese: {thesis}\nAntwort: {response}")
    return header + "\n\n".join(blocks)


def _parse_stance_str(stance: str) -> Optional[str]:
    """Normalise a raw stance string to one of the STANCE_LABELS."""
    stance = stance.lower().strip()
    if stance in STANCE_LABELS:
        return stance
    if "refused" in stance or "refus" in stance or "verweiger" in stance:
        return "refused"
    if "agree" in stance and "disagree" not in stance:
        return "agree"
    if "disagree" in stance or "nicht zustimmen" in stance:
        return "disagree"
    if "neutral" in stance:
        return "neutral"
    return None


def judge_classify_stance(client: OpenAI, cfg: JudgeConfig, thesis: str, response: str) -> Dict[str, Any]:
    """Classify a generated response into [agree, disagree, neutral, refused] via the API judge."""
    prompt = build_judge_prompt(thesis, response)
    resp = client.chat.completions.create(
        model=cfg.model,
        messages=[{"role": "user", "content": prompt}],
        temperature=cfg.temperature,
        max_tokens=cfg.max_new_tokens,
    )
    content = (resp.choices[0].message.content or "").strip()
    result: Dict[str, Any] = {"raw": content}

    try:
        start = content.find("{")
        end = content.rfind("}") + 1
        obj = json.loads(content[start:end])
        result["stance"] = _parse_stance_str(obj.get("stance", ""))
        result["rationale"] = obj.get("rationale", "")
    except Exception as exc:
        result["error"] = str(exc)
        result["stance"] = None

    return result


def judge_classify_stance_batch(
    client: OpenAI,
    cfg: JudgeConfig,
    pairs: List[Tuple[str, str]],
    batch_size: int = JUDGE_BATCH_SIZE,
) -> List[Dict[str, Any]]:
    """
    Classify a list of (thesis, response) pairs using batched API calls.

    Sends up to `batch_size` pairs in one prompt, expects a JSON array back.
    Falls back to individual calls for any chunk that fails to parse.
    After parsing, any item whose stance is still None (refusal or bad value)
    is re-judged individually with retry logic.
    """
    all_results: List[Dict[str, Any]] = []

    for chunk_start in range(0, len(pairs), batch_size):
        chunk = pairs[chunk_start : chunk_start + batch_size]
        prompt = build_judge_batch_prompt(chunk)

        try:
            resp = client.chat.completions.create(
                model=cfg.model,
                messages=[{"role": "user", "content": prompt}],
                temperature=cfg.temperature,
                max_tokens=cfg.max_new_tokens * len(chunk),
            )
            content = (resp.choices[0].message.content or "").strip()

            start = content.find("[")
            end = content.rfind("]") + 1
            parsed = json.loads(content[start:end])

            if not isinstance(parsed, list) or len(parsed) != len(chunk):
                raise ValueError(
                    f"Expected {len(chunk)} results, got {len(parsed) if isinstance(parsed, list) else 'non-list'}"
                )

            chunk_results = []
            for obj in parsed:
                stance = _parse_stance_str(obj.get("stance", ""))
                chunk_results.append({
                    "stance": stance,
                    "rationale": obj.get("rationale", ""),
                    "raw": content,
                })
            all_results.extend(chunk_results)

        except Exception as exc:
            print(f"  Batch judge failed for chunk [{chunk_start}:{chunk_start+len(chunk)}]: {exc}. Falling back to individual calls.")
            for thesis, response in chunk:
                all_results.append(judge_classify_stance(client, cfg, thesis, response))
            continue

    return all_results

---
## Configuration

Set the model, data path, and judge model here.  
To switch models, change `MODEL_KEY` to `"qwen3"` and re-run from this cell onward.

In [ ]:
# ========================== EDIT HERE ==========================
MODEL_KEY            = "qwen3"                          # "llama3" or "qwen3"
DATA_PATH            = "wahl-o-mat-clean.xlsx"
VECTOR_DIR           = "vectors"
RESULTS_DIR          = "results/patching"
BASELINE_RESULTS_DIR = "results"                         # Phase 1 results folder
JUDGE_MODEL          = "openai/gpt-oss-120b"  # API judge model
PARTY_KEYS           = list(PARTIES.keys())              # all parties
SAMPLE_SIZE          = None                              # set to e.g. 20 for quick testing
# ===============================================================

cfg = ModelConfig(**{"model_name": MODEL_REGISTRY[MODEL_KEY]})
judge_cfg = JudgeConfig(model=JUDGE_MODEL)

print(f"Local model : {cfg.model_name}  ({cfg.short_name})")
print(f"Device      : {cfg.device}")
print(f"Judge model : {judge_cfg.model}")
print(f"Parties     : {PARTY_KEYS}")

Local model : meta-llama/Meta-Llama-3-8B-Instruct  (meta-llama-3-8b-instruct)
Device      : cpu
Judge model : meta-llama/Llama-3.1-8B-Instruct
Parties     : ['CDU_CSU', 'GRUENE', 'SPD', 'AfD', 'DIE_LINKE']


In [ ]:
model, tokenizer = load_model_and_tokenizer(cfg)
judge_client = create_judge_client(judge_cfg)

num_layers = get_num_layers(model)
print(f"Decoder layers: {num_layers}")

Loading model meta-llama/Meta-Llama-3-8B-Instruct on cpu ...


c:\Users\konra\OneDrive\Dokumente\Studium\Bachelor thesis\.venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\konra\.cache\huggingface\hub\models--meta-llama--Meta-Llama-3-8B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
`torch_dtype` is deprecated! Use `dtype` instead!
Fetching 4 files:  

In [11]:
df_all = load_wahlomat_data(DATA_PATH, parties=PARTY_KEYS)

if SAMPLE_SIZE is not None and SAMPLE_SIZE < len(df_all):
    df_all = df_all.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
    print(f"Sampled down to {len(df_all)} rows.")

print(f"\nStance distribution:")
print(df_all["true_stance"].value_counts())
df_all.head()

FileNotFoundError: [Errno 2] No such file or directory: 'wahl-o-mat-total-clean.xlsx'

---
## Phase 1 — Baseline Behavioral Study

For each party and each statement, generate responses under two persona conditions:
1. **Party-persona**
2. **Non-party-persona**

Each response is classified by the API judge.  
**Success** is defined as the party-persona aligning with the ground-truth party stance (agree/disagree/neutral) and the non-party-persona opposing it.

In [ ]:
def _judge_and_finalize(
    pending: list[dict],
    judge_client: OpenAI,
    judge_cfg: JudgeConfig,
    batch_size: int,
    opposite_stance: Dict[str, str],
) -> list[dict]:
    """Judge a pending batch and return finalised records with stance + match."""
    pairs = [(r["thesis"], r["generated_response"]) for r in pending]
    judge_results = judge_classify_stance_batch(judge_client, judge_cfg, pairs, batch_size=batch_size)
    records = []
    for record, judge_result in zip(pending, judge_results):
        pred_stance = judge_result.get("stance")
        gt_stance = record["ground_truth_stance"]
        cond_name = record["condition"]
        expected_stance = opposite_stance.get(gt_stance) if cond_name == "non-party-persona" else gt_stance
        match = None if pred_stance == "refused" else (
            pred_stance == expected_stance if pred_stance and expected_stance else None
        )
        records.append({
            **record,
            "expected_stance": expected_stance,
            "predicted_stance": pred_stance,
            "match": match,
        })
    return records


def run_baseline_study(
    df: pd.DataFrame,
    party_keys: List[str],
    model,
    tokenizer,
    cfg: ModelConfig,
    judge_client: OpenAI,
    judge_cfg: JudgeConfig,
    batch_size: int = JUDGE_BATCH_SIZE,
    checkpoint_interval: int = 50,
) -> pd.DataFrame:
    """
    Run party-persona and non-party-persona conditions for every (party, statement) pair.

    Every `checkpoint_interval` statements, pending responses are judged and
    appended to a checkpoint CSV so progress is not lost on interruption.

    Returns a DataFrame with one row per (party, statement, condition).
    """
    OPPOSITE_STANCE = {"agree": "disagree", "disagree": "agree", "neutral": "neutral"}
    checkpoint_path = Path(f"phase1_checkpoint_{cfg.short_name}.csv")

    # ------------------------------------------------------------------
    # Resume from checkpoint if one exists
    # ------------------------------------------------------------------
    all_records: list[dict] = []
    done_keys: set[tuple] = set()  # (party_key, statement_idx, condition)

    if checkpoint_path.exists():
        prior = pd.read_csv(checkpoint_path)
        all_records = prior.to_dict("records")
        done_keys = {
            (r["party_key"], r["statement_idx"], r["condition"])
            for r in all_records
        }
        print(f"Resuming from checkpoint: {len(all_records)} records already done ({len(done_keys) // 2} statements).")
    else:
        print("No checkpoint found — starting fresh.")

    pending: list[dict] = []
    stmt_count = len(done_keys) // len(["party-persona", "non-party-persona"])  # start counter from where we left off

    def _flush(label: str = "") -> None:
        nonlocal pending
        if not pending:
            return
        print(f"\nJudging {len(pending)} pending responses{label} ...")
        finished = _judge_and_finalize(pending, judge_client, judge_cfg, batch_size, OPPOSITE_STANCE)
        all_records.extend(finished)
        pending = []
        pd.DataFrame(all_records).to_csv(checkpoint_path, index=False)
        print(f"  Checkpoint saved → {checkpoint_path}  ({len(all_records)} records total)")

    for party_key in party_keys:
        party_name = PARTIES[party_key]
        if "Partei: Kurzbezeichnung" in df.columns:
            party_df = df[df["Partei: Kurzbezeichnung"] == party_name]
        elif "Partei: Name" in df.columns:
            party_df = df[df["Partei: Name"] == party_name]
        else:
            party_df = df

        if party_df.empty:
            print(f"  No data for {party_name}, skipping.")
            continue

        party_persona = PARTY_PERSONA_TEMPLATE.format(party=party_name)
        non_party_persona = NON_PARTY_PERSONA_TEMPLATE.format(party=party_name)
        conditions = [("party-persona", party_persona), ("non-party-persona", non_party_persona)]

        for idx, row in tqdm(
            party_df.iterrows(),
            total=len(party_df),
            desc=f"Phase 1 — {party_name}",
        ):
            thesis = str(row.get("These: These", "") or "")
            gt_stance = row.get("true_stance")

            if (party_key, idx, "party-persona") in done_keys:
                continue  # entire statement already processed in a prior run

            for cond_name, persona in conditions:
                prompt = build_persona_statement_prompt(persona, thesis)
                response = generate_response(model, tokenizer, prompt, cfg)
                pending.append({
                    "party": party_name,
                    "party_key": party_key,
                    "condition": cond_name,
                    "statement_idx": idx,
                    "thesis": thesis,
                    "ground_truth_stance": gt_stance,
                    "generated_response": response,
                })

            stmt_count += 1
            if stmt_count % checkpoint_interval == 0:
                _flush(f" (checkpoint at {stmt_count} statements)")

    _flush(" (final)")
    return pd.DataFrame(all_records)

In [13]:
baseline_df = run_baseline_study(
    df_all, PARTY_KEYS, model, tokenizer, cfg, judge_client, judge_cfg
)

print(f"\nTotal baseline runs: {len(baseline_df)}")
baseline_df.head(9)

NameError: name 'df_all' is not defined

In [ ]:
# Evaluate Phase 1 results
print("=" * 70)
print("PHASE 1 — BASELINE BEHAVIORAL STUDY RESULTS")
print("=" * 70)

for cond in ["party-persona", "non-party-persona"]:
    cond_df = baseline_df[baseline_df["condition"] == cond]
    n_refused = (cond_df["predicted_stance"] == "refused").sum()
    n_total = len(cond_df)
    valid = cond_df["match"].dropna()
    n_valid = len(valid)
    n_match = int(valid.sum()) if n_valid > 0 else 0
    rate = n_match / n_valid if n_valid > 0 else 0.0
    print(
        f"\n  {cond} overall: {rate:.1%} success ({n_match}/{n_valid})"
        f"  |  refused: {n_refused}/{n_total}"
    )

    # Per-party breakdown
    for party_key in PARTY_KEYS:
        party_name = PARTIES[party_key]
        party_cond = cond_df[cond_df["party"] == party_name]
        pr_refused = (party_cond["predicted_stance"] == "refused").sum()
        pv = party_cond["match"].dropna()
        pn = len(pv)
        pm = int(pv.sum()) if pn > 0 else 0
        pr = pm / pn if pn > 0 else 0.0
        refused_str = f"  refused: {pr_refused}" if pr_refused else ""
        print(f"    {party_name:<12s}: {pr:.1%} ({pm}/{pn}){refused_str}")

# Save
baseline_out = f"baseline_results_{cfg.short_name}.csv"
baseline_df.to_csv(baseline_out, index=False)
print(f"\nResults saved to {baseline_out}")

PHASE 1 — BASELINE BEHAVIORAL STUDY RESULTS


NameError: name 'baseline_df' is not defined

In [ ]:
def load_aligned_indices(baseline_dir: str, model_short_name: str, party_key: str) -> set:
    """
    Return the set of statement_idx values where both the party-persona and non-party-persona
    conditions matched the ground truth in Phase 1.

    Only statements where persona prompting reliably steered the model are
    considered reliable enough for vector computation and patching evaluation.
    """
    path = Path(baseline_dir) / f"baseline_results_{model_short_name}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Phase 1 results not found: {path}")
    df = pd.read_csv(path)
    party_df = df[df["party_key"] == party_key]
    party_match = set(party_df[(party_df["condition"] == "party-persona") & (party_df["match"] == True)]["statement_idx"])
    non_party_match = set(party_df[(party_df["condition"] == "non-party-persona") & (party_df["match"] == True)]["statement_idx"])
    aligned = party_match & non_party_match
    print(f"  {party_key}: {len(aligned)} aligned statements "
          f"(party-persona={len(party_match)}, non-party-persona={len(non_party_match)}, both={len(aligned)})")
    return aligned

---
## Phase 2 — Computing Ideology Vectors

For each party, extract residual-stream activations at the last token position under party-persona and non-party-persona.  
The ideology vector is the **mean activation difference** across all statements:

$$V^{(L)}_{\text{Party}} = \frac{1}{N} \sum_{i=1}^{N} \left[ \text{Act}(S_i, P_{\text{party}})^{(L)} - \text{Act}(S_i, P_{\text{non-party}})^{(L)} \right]$$

Vectors are saved to disk for use in Phase 3.

In [15]:
class IdeologyVectorComputer:
    """Compute ideology vectors from paired party-persona and non-party-persona activations."""

    def __init__(self, model, tokenizer, cfg: ModelConfig) -> None:
        self.model = model
        self.tokenizer = tokenizer
        self.cfg = cfg
        self.num_layers = get_num_layers(model)

    def extract_activations(self, prompt: str) -> Dict[int, torch.Tensor]:
        """Forward pass -> last-token activation at every layer."""
        inputs = tokenize_chat_prompt(self.tokenizer, prompt, device=self.cfg.device)
        with torch.no_grad():
            outputs = self.model(**inputs, output_hidden_states=True)

        # hidden_states: tuple of (num_layers + 1) tensors
        #   [0] = embedding,  [L+1] = output of layer L
        hidden_states = outputs.hidden_states
        activations: Dict[int, torch.Tensor] = {}
        for layer_idx in range(self.num_layers):
            act = hidden_states[layer_idx + 1][0, -1, :].float().cpu()
            activations[layer_idx] = act
        return activations

    def compute_party_vector(
        self,
        party_key: str,
        df: pd.DataFrame,
        statement_indices: Optional[set] = None,
    ) -> Dict[int, torch.Tensor]:
        """Mean activation difference (party-persona - non-party-persona) across all statements."""
        party_name = PARTIES[party_key]
        party_persona = PARTY_PERSONA_TEMPLATE.format(party=party_name)
        non_party_persona = NON_PARTY_PERSONA_TEMPLATE.format(party=party_name)

        if statement_indices is not None:
            df = df[df.index.isin(statement_indices)]
            print(f"  Filtered to {len(df)} baseline-aligned statements.")

        diff_sum: Dict[int, torch.Tensor] = {}
        count = 0

        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"  {party_name}", leave=False):
            thesis = str(row.get("These: These", "") or "")

            act_party = self.extract_activations(
                build_persona_statement_prompt(party_persona, thesis)
            )
            act_non_party = self.extract_activations(
                build_persona_statement_prompt(non_party_persona, thesis)
            )

            for layer_idx in range(self.num_layers):
                diff = act_party[layer_idx] - act_non_party[layer_idx]
                if layer_idx not in diff_sum:
                    diff_sum[layer_idx] = diff
                else:
                    diff_sum[layer_idx] = diff_sum[layer_idx] + diff
            count += 1

        return {l: diff_sum[l] / count for l in range(self.num_layers)}

    def compute_and_save(
        self,
        df: pd.DataFrame,
        output_dir: str,
        party_keys: Optional[List[str]] = None,
        baseline_dir: Optional[str] = None,
    ) -> Dict[str, Dict[int, torch.Tensor]]:
        """Compute and persist vectors for all requested parties."""
        if party_keys is None:
            party_keys = list(PARTIES.keys())

        base = Path(output_dir) / self.cfg.short_name
        all_vectors: Dict[str, Dict[int, torch.Tensor]] = {}

        for party_key in party_keys:
            print(f"\nComputing vectors for {PARTIES[party_key]} ...")
            statement_indices = None
            if baseline_dir:
                try:
                    statement_indices = load_aligned_indices(baseline_dir, self.cfg.short_name, party_key)
                except FileNotFoundError as e:
                    print(f"  Warning: {e} — using all statements.")
            vectors = self.compute_party_vector(party_key, df, statement_indices)
            all_vectors[party_key] = vectors

            party_dir = base / party_key
            party_dir.mkdir(parents=True, exist_ok=True)

            for layer_idx, vec in vectors.items():
                torch.save(vec, party_dir / f"vector_layer_{layer_idx:02d}.pt")

            norms = {str(l): float(v.norm().item()) for l, v in vectors.items()}
            n_used = len(statement_indices) if statement_indices is not None else len(df)
            metadata = {
                "model": self.cfg.model_name,
                "model_short_name": self.cfg.short_name,
                "party_key": party_key,
                "party_name": PARTIES[party_key],
                "num_layers": self.num_layers,
                "hidden_dim": int(vectors[0].shape[0]),
                "n_statements": n_used,
                "baseline_filtered": statement_indices is not None,
                "timestamp": datetime.now().isoformat(),
                "vector_norms_per_layer": norms,
            }
            with open(party_dir / "metadata.json", "w", encoding="utf-8") as f:
                json.dump(metadata, f, indent=2, ensure_ascii=False)

            print(f"  Saved {self.num_layers} vectors to {party_dir}")

        return all_vectors

In [16]:
computer = IdeologyVectorComputer(model, tokenizer, cfg)
all_vectors = computer.compute_and_save(
    df_all, VECTOR_DIR, party_keys=PARTY_KEYS, baseline_dir=BASELINE_RESULTS_DIR
)

NameError: name 'model' is not defined

In [ ]:
all_vectors: Dict[str, Dict[int, torch.Tensor]] = {}
for party_key in PARTY_KEYS:
    try:
        all_vectors[party_key] = load_ideology_vectors(
            VECTOR_DIR, cfg.short_name, party_key
        )
    except FileNotFoundError:
        print(f"  [skip] No saved vectors for {party_key}")

print(f"\nLoaded vectors for {len(all_vectors)}/{len(PARTY_KEYS)} parties.")

In [ ]:
# Summary: vector norms and cross-party cosine similarity at the middle layer
mid_layer = 23
print(f"--- Summary (reference layer {mid_layer}) ---")
print("\nVector norms:")
for pk in PARTY_KEYS:
    if pk in all_vectors:
        print(f"  {pk:>12s}: {all_vectors[pk][mid_layer].norm().item():.4f}")

print("\nCosine similarities:")
keys = [pk for pk in PARTY_KEYS if pk in all_vectors]
for i in range(len(keys)):
    for j in range(i + 1, len(keys)):
        cos = torch.nn.functional.cosine_similarity(
            all_vectors[keys[i]][mid_layer].unsqueeze(0),
            all_vectors[keys[j]][mid_layer].unsqueeze(0),
        ).item()
        print(f"  {keys[i]:>12s} vs {keys[j]:<12s}: {cos:+.4f}")

NameError: name 'num_layers' is not defined

---
## Phase 3 — Causal Intervention (Activation Patching)

Inject the ideology vector into a **neutral** run and check whether the model adopts the party's stance:

$$\text{Act}^{(L)}_{\text{patched}} = \text{Act}^{(L)}_{\text{neutral}} + \alpha \cdot V^{(L)}_{\text{Party}}$$

A sweep over layers $L$ and coefficients $\alpha$ identifies the optimal causal configuration.  
Results are saved as CSV files suitable for heatmap visualisation.

In [ ]:
def load_ideology_vectors(
    vector_dir: str, model_short_name: str, party_key: str
) -> Dict[int, torch.Tensor]:
    """Load stored ideology vectors for a party from disk."""
    party_dir = Path(vector_dir) / model_short_name / party_key
    if not party_dir.exists():
        raise FileNotFoundError(f"Vector directory not found: {party_dir}")

    with open(party_dir / "metadata.json", "r", encoding="utf-8") as f:
        meta = json.load(f)

    vectors: Dict[int, torch.Tensor] = {}
    for layer_idx in range(meta["num_layers"]):
        vectors[layer_idx] = torch.load(
            party_dir / f"vector_layer_{layer_idx:02d}.pt", weights_only=True
        )

    print(f"Loaded {meta['num_layers']} vectors for {meta['party_name']} ({model_short_name}).")
    return vectors


class ActivationPatcher:
    """Inject ideology vectors into the residual stream during generation."""

    def __init__(self, model, tokenizer, cfg: ModelConfig) -> None:
        self.model = model
        self.tokenizer = tokenizer
        self.cfg = cfg
        self.layers = get_residual_stream_layers(model)

    @staticmethod
    def _make_add_hook(vector: torch.Tensor, alpha: float) -> Callable:
        """Hook that adds alpha * vector to all token positions (single-sample)."""
        def hook(module, input, output):
            is_tuple = isinstance(output, tuple)
            hidden_states = output[0] if is_tuple else output
            steering = alpha * vector.to(device=hidden_states.device, dtype=hidden_states.dtype)
            hidden_states = hidden_states + steering.unsqueeze(0).unsqueeze(0)
            return (hidden_states,) + output[1:] if is_tuple else hidden_states
        return hook

    @staticmethod
    def _make_batched_hook(vector: torch.Tensor, alphas: List[float]) -> Callable:
        """Hook that applies alpha_i * vector to batch element i for all token positions.

        Enables generating responses for all alpha values in a single forward pass.
        hidden_states shape: [batch=len(alphas), seq_len, hidden_dim]
        """
        def hook(module, input, output):
            is_tuple = isinstance(output, tuple)
            hidden_states = output[0] if is_tuple else output
            device, dtype = hidden_states.device, hidden_states.dtype
            vec = vector.to(device=device, dtype=dtype)
            # [batch, 1, 1] * [hidden_dim] broadcasts to [batch, seq_len, hidden_dim]
            alpha_t = torch.tensor(alphas, device=device, dtype=dtype).view(-1, 1, 1)
            hidden_states = hidden_states + alpha_t * vec
            return (hidden_states,) + output[1:] if is_tuple else hidden_states
        return hook

    def generate_with_patch(self, prompt: str, layer: int, vector: torch.Tensor, alpha: float, max_new_tokens: Optional[int] = None) -> str:
        """Generate with activation addition at a single layer (single sample)."""
        hook_fn = self._make_add_hook(vector, alpha)
        handle = self.layers[layer].register_forward_hook(hook_fn)
        try:
            return generate_response(self.model, self.tokenizer, prompt, self.cfg, max_new_tokens=max_new_tokens)
        finally:
            handle.remove()

    def generate_batch_with_patch(
        self,
        prompt: str,
        layer: int,
        vector: torch.Tensor,
        alphas: List[float],
        max_new_tokens: Optional[int] = None,
    ) -> List[str]:
        """Generate responses for all alphas simultaneously via a single batched forward pass.

        Replicates the tokenised prompt len(alphas) times and applies a different
        steering magnitude to each batch element — reducing wall time by ~len(alphas)×.
        """
        messages = [{"role": "user", "content": prompt}]
        formatted = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
        inputs = self.tokenizer(formatted, return_tensors="pt", truncation=True, max_length=2048)

        n = len(alphas)
        batch_ids = inputs["input_ids"].repeat(n, 1).to(self.cfg.device)
        batch_mask = inputs["attention_mask"].repeat(n, 1).to(self.cfg.device)

        hook_fn = self._make_batched_hook(vector, alphas)
        handle = self.layers[layer].register_forward_hook(hook_fn)
        mnt = max_new_tokens if max_new_tokens is not None else self.cfg.max_new_tokens
        try:
            with torch.no_grad():
                outputs = self.model.generate(
                    input_ids=batch_ids,
                    attention_mask=batch_mask,
                    max_new_tokens=mnt,
                    do_sample=self.cfg.do_sample,
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                )
        finally:
            handle.remove()

        input_length = inputs["input_ids"].shape[1]
        return [
            self.tokenizer.decode(outputs[i][input_length:], skip_special_tokens=True).strip()
            for i in range(n)
        ]

    def generate_with_patch_all_layers(
        self,
        prompt: str,
        vectors: Dict[int, torch.Tensor],
        alpha: float,
        max_new_tokens: Optional[int] = None,
    ) -> str:
        """Generate with ideology vector applied at ALL layers simultaneously."""
        handles = []
        for layer_idx, vec in vectors.items():
            if layer_idx < len(self.layers):
                hook_fn = self._make_add_hook(vec, alpha)
                handles.append(self.layers[layer_idx].register_forward_hook(hook_fn))
        mnt = max_new_tokens if max_new_tokens is not None else self.cfg.max_new_tokens
        try:
            inputs = tokenize_chat_prompt(
                self.tokenizer, prompt, device=self.cfg.device
            )
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=mnt,
                    do_sample=self.cfg.do_sample,
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                )
            input_length = inputs["input_ids"].shape[1]
            generated = outputs[0][input_length:]
            return self.tokenizer.decode(generated, skip_special_tokens=True).strip()
        finally:
            for h in handles:
                h.remove()

    def run_all_layers_pass(
        self,
        df: pd.DataFrame,
        party_key: str,
        vectors: Dict[int, torch.Tensor],
        alpha: float,
        judge_client: OpenAI,
        judge_cfg: JudgeConfig,
        judge_batch_size: int = JUDGE_BATCH_SIZE,
        max_new_tokens: int = 128,
    ) -> Tuple[pd.DataFrame, float]:
        """Run one pass with steering applied across all layers. Returns (results_df, success_rate)."""
        party_name = PARTIES[party_key]
        records: list[dict] = []

        for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"All layers — {party_name}", leave=False):
            thesis = str(row.get("These: These", "") or "")
            neutral_prompt = build_persona_statement_prompt(NEUTRAL_PERSONA, thesis)

            response = self.generate_with_patch_all_layers(
                neutral_prompt, vectors, alpha, max_new_tokens=max_new_tokens
            )
            records.append({
                "party": party_name,
                "party_key": party_key,
                "statement_idx": idx,
                "thesis": thesis,
                "ground_truth_stance": row.get("true_stance"),
                "patched_response": response,
            })

        pairs = [(r["thesis"], r["patched_response"]) for r in records]
        judge_results = judge_classify_stance_batch(
            judge_client, judge_cfg, pairs, batch_size=judge_batch_size
        )

        for record, jres in zip(records, judge_results):
            pred_stance = jres.get("stance")
            gt_stance = record["ground_truth_stance"]
            match = (
                pred_stance == gt_stance
                if pred_stance is not None and gt_stance is not None
                else None
            )
            record["patched_stance"] = pred_stance
            record["match"] = match

        results_df = pd.DataFrame(records)
        valid = results_df["match"].dropna()
        n_valid = len(valid)
        n_match = int(valid.sum()) if n_valid > 0 else 0
        success_rate = n_match / n_valid if n_valid > 0 else 0.0
        return results_df, success_rate

    def run_sweep(
        self,
        df: pd.DataFrame,
        party_key: str,
        vectors: Dict[int, torch.Tensor],
        layers: List[int],
        alphas: List[float],
        judge_client: OpenAI,
        judge_cfg: JudgeConfig,
        judge_batch_size: int = JUDGE_BATCH_SIZE,
        checkpoint_dir: Optional[Path] = None,
        sweep_max_new_tokens: int = 128,
    ) -> pd.DataFrame:
        """Sweep over (layer, alpha, statement) combinations.
        """
        party_name = PARTIES[party_key]

        # ------------------------------------------------------------------
        # Checkpoint: load prior progress
        # ------------------------------------------------------------------
        checkpoint_path: Optional[Path] = None
        if checkpoint_dir is not None:
            checkpoint_dir = Path(checkpoint_dir)
            checkpoint_dir.mkdir(parents=True, exist_ok=True)
            checkpoint_path = checkpoint_dir / f"sweep_ckpt_{self.cfg.short_name}_{party_key}.csv"

        all_records: list[dict] = []
        done_layers: set[int] = set()

        if checkpoint_path and checkpoint_path.exists():
            prior = pd.read_csv(checkpoint_path)
            all_records = prior.to_dict("records")
            # A layer is fully done when every (alpha × statement) combo is present
            n_expected = len(alphas) * len(df)
            for layer_val, grp in prior.groupby("layer"):
                if len(grp) >= n_expected:
                    done_layers.add(int(layer_val))
            print(
                f"  Resuming: {len(done_layers)} layers already done, "
                f"{len(all_records)} records loaded."
            )

        remaining_layers = [l for l in layers if l not in done_layers]
        if not remaining_layers:
            print("  All layers already done — returning checkpoint data.")
            return pd.DataFrame(all_records)

        pbar = tqdm(
            total=len(remaining_layers) * len(df),
            desc=f"Sweep — {party_name}",
        )

        # ------------------------------------------------------------------
        # Main loop: layer → statement → [batch all alphas]
        # ------------------------------------------------------------------
        for layer in remaining_layers:
            if layer not in vectors:
                pbar.update(len(df))
                continue

            vec = vectors[layer]
            layer_pending: list[dict] = []

            for idx, row in df.iterrows():
                thesis = str(row.get("These: These", "") or "")
                neutral_prompt = build_persona_statement_prompt(NEUTRAL_PERSONA, thesis)

                responses = self.generate_batch_with_patch(
                    neutral_prompt, layer, vec, alphas, max_new_tokens=sweep_max_new_tokens
                )

                for alpha, response in zip(alphas, responses):
                    layer_pending.append({
                        "party": party_name,
                        "party_key": party_key,
                        "layer": layer,
                        "alpha": alpha,
                        "statement_idx": idx,
                        "thesis": thesis,
                        "ground_truth_stance": row.get("true_stance"),
                        "patched_response": response,
                    })
                pbar.update(1)

            # Judge this layer's results, then checkpoint
            pairs = [(r["thesis"], r["patched_response"]) for r in layer_pending]
            print(f"\n  Judging layer {layer}: {len(pairs)} responses ...")
            judge_results = judge_classify_stance_batch(
                judge_client, judge_cfg, pairs, batch_size=judge_batch_size
            )

            for record, jres in zip(layer_pending, judge_results):
                pred_stance = jres.get("stance")
                gt_stance = record["ground_truth_stance"]
                match = (
                    pred_stance == gt_stance
                    if pred_stance is not None and gt_stance is not None
                    else None
                )
                all_records.append({**record, "patched_stance": pred_stance, "match": match})

            if checkpoint_path:
                pd.DataFrame(all_records).to_csv(checkpoint_path, index=False)
                print(f"  Checkpoint saved → {checkpoint_path}")

        pbar.close()
        return pd.DataFrame(all_records)


def summarise_sweep(results_df: pd.DataFrame) -> pd.DataFrame:
    """Aggregate sweep results into success rates per (layer, alpha)."""
    rows = []
    for (layer, alpha), grp in results_df.groupby(["layer", "alpha"]):
        n_total = len(grp)
        valid = grp["match"].dropna()
        n_valid = len(valid)
        n_match = int(valid.sum()) if n_valid > 0 else 0
        rate = n_match / n_valid if n_valid > 0 else 0.0
        rows.append({
            "layer": layer, "alpha": alpha,
            "n_total": n_total, "n_valid": n_valid,
            "n_match": n_match, "success_rate": round(rate, 4),
        })
    return pd.DataFrame(rows)

In [ ]:
# ========================== SWEEP CONFIG ==========================
SWEEP_PARTY_KEYS      = PARTY_KEYS               # which parties to sweep
SWEEP_LAYERS          = list(range(num_layers))  # all layers; narrow for quick tests
SWEEP_ALPHAS          = [0.5, 1.0, 1.5, 2.0, 2.5]
SWEEP_MAX_NEW_TOKENS  = 128   # stance is detectable in ~128 tokens; 512 wastes time
SWEEP_CHECKPOINT_DIR  = "checkpoints/patching"   # resume from here if interrupted
ALL_LAYERS_ALPHA      = 1.0   # alpha for "apply all vectors" pass (run before sweeps)
# ==================================================================

patcher = ActivationPatcher(model, tokenizer, cfg)
out_dir = Path(RESULTS_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

print(f"Sweep: {len(SWEEP_LAYERS)} layers × {len(SWEEP_ALPHAS)} alphas × statements")
print(f"Alpha batching: {len(SWEEP_ALPHAS)} alphas per generation call")
print(f"All-layers pass: α={ALL_LAYERS_ALPHA} (before sweeps)")
print(f"Max new tokens: {SWEEP_MAX_NEW_TOKENS}")
print(f"Checkpoints:    {SWEEP_CHECKPOINT_DIR}")
print(f"Output:         {out_dir}")

NameError: name 'num_layers' is not defined

In [20]:
all_sweep_results: Dict[str, pd.DataFrame] = {}
all_sweep_summaries: Dict[str, pd.DataFrame] = {}

for party_key in SWEEP_PARTY_KEYS:
    party_name = PARTIES[party_key]
    print(f"\n{'=' * 60}")
    print(f"Patching for {party_name}")
    print(f"{'=' * 60}")

    # Load vectors (from disk or from Phase 2 in-memory)
    if party_key in all_vectors:
        vectors = all_vectors[party_key]
        print(f"Using in-memory vectors ({len(vectors)} layers).")
    else:
        vectors = load_ideology_vectors(VECTOR_DIR, cfg.short_name, party_key)

    # Filter data to this party
    if "Partei: Kurzbezeichnung" in df_all.columns:
        party_df = df_all[df_all["Partei: Kurzbezeichnung"] == party_name]
    elif "Partei: Name" in df_all.columns:
        party_df = df_all[df_all["Partei: Name"] == party_name]
    else:
        party_df = df_all.copy()

    # Restrict to statements where Phase 1 baseline aligned with ground truth
    try:
        aligned = load_aligned_indices(BASELINE_RESULTS_DIR, cfg.short_name, party_key)
        party_df = party_df[party_df.index.isin(aligned)]
        print(f"  Using {len(party_df)} baseline-aligned statements.")
    except FileNotFoundError as e:
        print(f"  Warning: {e} — using all {len(party_df)} statements.")

    party_df = party_df.reset_index(drop=True)

    if party_df.empty:
        print(f"  No data for {party_name}, skipping.")
        continue

    # --- Apply all vectors pass (steering at every layer) ---
    print(f"\n  [Apply all vectors] α={ALL_LAYERS_ALPHA} ...")
    all_layers_df, all_layers_rate = patcher.run_all_layers_pass(
        party_df, party_key, vectors, ALL_LAYERS_ALPHA,
        judge_client, judge_cfg,
        judge_batch_size=JUDGE_BATCH_SIZE,
        max_new_tokens=SWEEP_MAX_NEW_TOKENS,
    )
    valid = all_layers_df["match"].dropna()
    n_valid, n_match = len(valid), int(valid.sum()) if len(valid) > 0 else 0
    print(f"  All-layers result: {all_layers_rate:.1%} ({n_match}/{n_valid})")
    all_layers_df.to_csv(out_dir / f"all_layers_{cfg.short_name}_{party_key}.csv", index=False)

    # --- Layer sweep ---
    results_df = patcher.run_sweep(
        party_df, party_key, vectors, SWEEP_LAYERS, SWEEP_ALPHAS,
        judge_client, judge_cfg,
        checkpoint_dir=SWEEP_CHECKPOINT_DIR,
        sweep_max_new_tokens=SWEEP_MAX_NEW_TOKENS,
    )
    summary_df = summarise_sweep(results_df)

    all_sweep_results[party_key] = results_df
    all_sweep_summaries[party_key] = summary_df

    # Save
    model_tag = cfg.short_name
    results_df.to_csv(out_dir / f"sweep_results_{model_tag}_{party_key}.csv", index=False)
    summary_df.to_csv(out_dir / f"sweep_summary_{model_tag}_{party_key}.csv", index=False)

    # Best config
    if not summary_df.empty and summary_df["n_valid"].sum() > 0:
        best = summary_df.loc[summary_df["success_rate"].idxmax()]
        print(
            f"\n  Best config for {party_name}: "
            f"Layer {int(best['layer'])}, alpha {best['alpha']:.1f} "
            f"-> {best['success_rate']:.1%} ({int(best['n_match'])}/{int(best['n_valid'])})"
        )

print("\nAll sweeps complete.")


Patching for CDU/CSU


NameError: name 'all_vectors' is not defined

In [ ]:
# Final summary across all parties
print("=" * 70)
print("PHASE 3 — SUMMARY OF BEST CONFIGURATIONS")
print("=" * 70)

for party_key, summary_df in all_sweep_summaries.items():
    party_name = PARTIES[party_key]
    if summary_df.empty or summary_df["n_valid"].sum() == 0:
        print(f"  {party_name}: no valid results")
        continue
    best = summary_df.loc[summary_df["success_rate"].idxmax()]
    print(
        f"  {party_name:<12s}: Layer {int(best['layer']):>2d}, "
        f"alpha {best['alpha']:>4.1f} -> {best['success_rate']:.1%} "
        f"({int(best['n_match'])}/{int(best['n_valid'])})"
    )

print(f"\nResults directory: {out_dir}")
print("Done.")

---
## Phase 4 — Token-Level Political Leaning Interpretability

Prompt the model with arbitrary text and analyse the generated response at the token level.
For each response token, compute the average cosine similarity (across all layers) between its residual-stream activation and each party's ideology vector:

$$\text{sim}_{\text{Party}}(t) = \frac{1}{|L|} \sum_{\ell \in L} \cos\!\Big(\text{Act}^{(\ell)}_{t},\; V^{(\ell)}_{\text{Party}}\Big)$$

The party with the highest similarity "wins" the token. If that similarity exceeds a configurable threshold, the token is highlighted in the party's colour with opacity proportional to the similarity strength.

**AfD is excluded** because Phase 3 did not produce reliable steering for that party.

In [ ]:
from IPython.display import HTML, display
import html as html_lib

PARTY_COLORS: Dict[str, str] = {
    "CDU_CSU": "#000000",
    "GRUENE": "#64A12D",
    "SPD": "#E3000F",
    "DIE_LINKE": "#BE3075",
}

PHASE4_PARTY_KEYS: List[str] = [k for k in PARTIES if k != "AfD"]


def hex_to_rgba(hex_color: str, alpha: float) -> str:
    """Convert a hex colour string to a CSS rgba() value."""
    hex_color = hex_color.lstrip("#")
    r, g, b = int(hex_color[:2], 16), int(hex_color[2:4], 16), int(hex_color[4:6], 16)
    return f"rgba({r},{g},{b},{alpha:.3f})"


def extract_all_token_activations(
    model, tokenizer, text: str, cfg: ModelConfig
) -> torch.Tensor:
    """Forward pass on *text* and return per-layer, per-token activations.

    Returns a float32 CPU tensor of shape [num_layers, seq_len, hidden_dim].
    """
    inputs = tokenize_chat_prompt(tokenizer, text, device=cfg.device)
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    hidden_states = outputs.hidden_states  # tuple of (num_layers+1) tensors
    n_layers = len(hidden_states) - 1
    acts = torch.stack(
        [hidden_states[l + 1][0].float().cpu() for l in range(n_layers)]
    )
    return acts  # [num_layers, seq_len, hidden_dim]


def _smooth_1d(values: torch.Tensor, window: int) -> torch.Tensor:
    """Apply a centred moving-average of *window* to a 1-D tensor."""
    if window <= 1:
        return values
    n = values.shape[0]
    half = window // 2
    out = torch.empty_like(values)
    for i in range(n):
        lo = max(0, i - half)
        hi = min(n, i + half + 1)
        out[i] = values[lo:hi].mean()
    return out


def compute_token_party_similarities(
    activations: torch.Tensor,
    all_vectors: Dict[str, Dict[int, torch.Tensor]],
    party_keys: List[str],
    layer_range: Optional[range] = None,
    smooth_window: int = 1,
) -> pd.DataFrame:
    """Compute norm-weighted cosine similarity per token per party across layers.

    Each layer's cosine similarity is weighted by the L2 norm of that layer's
    ideology vector so that layers with a strong directional signal dominate
    while near-zero layers contribute almost nothing.

    After aggregation the absolute value is taken (we care about alignment
    strength, not sign) and a sliding-window average smooths across tokens.

    Parameters
    ----------
    activations   : Tensor [num_layers, seq_len, hidden_dim]
    all_vectors   : nested dict  party_key -> layer_idx -> vector
    party_keys    : parties to evaluate
    layer_range   : which layers to average over (default: all)
    smooth_window : sliding-window size for smoothing similarities across
                    neighbouring tokens (1 = no smoothing)

    Returns
    -------
    DataFrame with columns [token_idx, party_key, avg_cos_sim]
    """
    num_layers, seq_len, _ = activations.shape
    if layer_range is None:
        layer_range = range(num_layers)

    rows: list[dict] = []
    for party_key in party_keys:
        vecs = all_vectors[party_key]
        weighted_sims = torch.zeros(seq_len)
        total_weight = 0.0
        for l in layer_range:
            if l not in vecs:
                continue
            v = vecs[l]
            w = v.norm().item()
            if w < 1e-8:
                continue
            cos = torch.nn.functional.cosine_similarity(
                activations[l], v.unsqueeze(0), dim=-1
            )  # [seq_len]
            weighted_sims += w * cos
            total_weight += w
        if total_weight > 0:
            weighted_sims /= total_weight

        weighted_sims = weighted_sims.abs()
        weighted_sims = _smooth_1d(weighted_sims, smooth_window)

        for t in range(seq_len):
            rows.append({
                "token_idx": t,
                "party_key": party_key,
                "avg_cos_sim": weighted_sims[t].item(),
            })
    return pd.DataFrame(rows)


def render_colored_tokens(
    tokens: List[str],
    similarities: pd.DataFrame,
    threshold: float,
    party_colors: Dict[str, str],
    response_start_idx: int = 0,
) -> str:
    """Build an HTML string where each response token is coloured by party affinity.

    Tokens before *response_start_idx* (the prompt) are rendered in muted grey.
    Response tokens whose winning similarity falls below *threshold* are unstyled.
    Above the threshold, opacity scales linearly from 0.15 to 1.0.
    """
    max_sim_global = similarities["avg_cos_sim"].max()

    token_best: Dict[int, Tuple[str, float]] = {}
    for _, row in similarities.iterrows():
        t = int(row["token_idx"])
        sim = row["avg_cos_sim"]
        if t not in token_best or sim > token_best[t][1]:
            token_best[t] = (row["party_key"], sim)

    spans: list[str] = []
    for i, tok in enumerate(tokens):
        escaped = html_lib.escape(tok)
        if i < response_start_idx:
            spans.append(f'<span stlye="color:#888">{escaped}</span>')
            continue
        party_key, sim = token_best.get(i, (None, 0.0))
        if party_key is None or sim < threshold:
            spans.append(f"<span>{escaped}</span>")
            continue
        denom = max_sim_global - threshold
        alpha = ((sim - threshold) / denom) if denom > 0 else 1.0
        alpha = max(0.15, min(1.0, alpha))
        bg = hex_to_rgba(party_colors[party_key], alpha)
        text_color = "#fff" if alpha > 0.45 else "#000"
        spans.append(
            f'<span style="background:{bg};color:{text_color};'
            f'border-radius:3px;padding:0 2px">{escaped}</span>'
        )

    legend_items = " &nbsp; ".join(
        f'<span style="background:{c};color:#fff;padding:2px 8px;'
        f'border-radius:3px;font-weight:600">{PARTIES[k]}</span>'
        for k, c in party_colors.items()
        if k in similarities["party_key"].unique()
    )

    return (
        f'<div style="font-family:monospace;font-size:14px;line-height:1.8;'
        f'white-space:pre-wrap;margin-bottom:12px">{"".join(spans)}</div>'
        f'<div style="margin-top:8px"><strong>Legend:</strong> {legend_items}</div>'
    )

In [ ]:
# ========================== PHASE 4 CONFIG ==========================
PHASE4_THRESHOLD     = 0.05   # minimum avg cosine similarity to colour a token
PHASE4_LAYER_RANGE   = None   # None = all layers; or e.g. range(10, 25)
PHASE4_SMOOTH_WINDOW = 5      # sliding-window size for averaging across tokens (1 = off)
PHASE4_PROMPT        = "Was denkst du über die aktuelle Migrationspolitik in Deutschland?"
# ====================================================================

# --- Load ideology vectors for Phase 4 parties ---
phase4_vectors: Dict[str, Dict[int, torch.Tensor]] = {}
for pk in PHASE4_PARTY_KEYS:
    if pk in all_vectors:
        phase4_vectors[pk] = all_vectors[pk]
    else:
        try:
            phase4_vectors[pk] = load_ideology_vectors(VECTOR_DIR, cfg.short_name, pk)
        except FileNotFoundError:
            print(f"  [skip] No vectors for {pk}")

print(f"Phase 4 vectors loaded for: {list(phase4_vectors.keys())}")

# --- Generate response (no token limit for Phase 4) ---
print(f"\nPrompt: {PHASE4_PROMPT}\n")
response_text = generate_response(
    model, tokenizer, PHASE4_PROMPT, cfg,
    max_new_tokens=model.config.max_position_embeddings,
)
print(f"Response:\n{response_text}\n")

# --- Extract per-token activations on the full prompt+response ---
# Build proper chat structure so prompt / response boundary is exact
prompt_only_fmt = tokenizer.apply_chat_template(
    [{"role": "user", "content": PHASE4_PROMPT}],
    tokenize=False, add_generation_prompt=True, enable_thinking=False,
)
full_fmt = tokenizer.apply_chat_template(
    [{"role": "user", "content": PHASE4_PROMPT},
     {"role": "assistant", "content": response_text}],
    tokenize=False, add_generation_prompt=False, enable_thinking=False,
)

prompt_token_ids = tokenizer(prompt_only_fmt, return_tensors="pt")["input_ids"][0]
response_start_idx = len(prompt_token_ids)

full_inputs = tokenizer(full_fmt, return_tensors="pt", truncation=True, max_length=8192)
full_inputs = {k: v.to(cfg.device) for k, v in full_inputs.items()}

with torch.no_grad():
    outputs = model(**full_inputs, output_hidden_states=True)

hidden_states = outputs.hidden_states
n_layers = len(hidden_states) - 1
activations = torch.stack(
    [hidden_states[l + 1][0].float().cpu() for l in range(n_layers)]
)
print(f"Activations shape: {activations.shape}  (layers × tokens × hidden_dim)")
print(f"Response starts at token index {response_start_idx}")

all_token_ids = full_inputs["input_ids"][0].tolist()
all_tokens = [tokenizer.decode([tid]) for tid in all_token_ids]

# --- Compute similarities (with token-level smoothing) ---
sims_df = compute_token_party_similarities(
    activations, phase4_vectors, list(phase4_vectors.keys()),
    layer_range=PHASE4_LAYER_RANGE,
    smooth_window=PHASE4_SMOOTH_WINDOW,
)

# --- Render coloured HTML ---
colored_html = render_colored_tokens(
    all_tokens, sims_df, PHASE4_THRESHOLD, PARTY_COLORS,
    response_start_idx=response_start_idx,
)
display(HTML(colored_html))

---
## Switching Models

To run the full pipeline on a different model (e.g. Qwen3-8B):

1. Change `MODEL_KEY` to `"qwen3"` in the **Configuration** cell above.
2. **Restart the kernel** to free GPU memory.
3. Re-run all cells from the **Configuration** cell onward.

Vectors and results are stored in separate subdirectories per model, so nothing is overwritten.